In [1]:
import sys
from pathlib import Path

project_root = Path(
    r"C:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\GlobalPPREstimation"
)
module_dir = str(project_root / "tools" / "scientific_code" / "PPREstimation")

if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

import pandas as pd
from PPRCalculator import PPRCalculator
from ModelData import ModelData

# Loading

In [2]:
model_path = (
    "../model.json"
)

model = ModelData(
    model_path,
    model_name="Guinea Current",
    model_year="1998",
)

# Inspect or edit these before constructing the calculator:
# model.groups_data
# model.DC
# model.det_fate

In [7]:
calculator = PPRCalculator.from_modeldata(
    modeldata=model,
    underdetermined=True,     # Estimate missing flows using LIM if True
    zero_catch=True,           # Replace missing catch with zero
    zero_biomass_accum=False,   # Replace missing biomass accumulation with zero
    default_gs=True,           # Supply GS=0.2 for regular groups missing GS
    weight_flow=1.0,           # LIM minimum-flow penalty weight
    weight_guess=1.0,          # LIM guess-deviation penalty weight
    normalize_DC=True,         # Normalize consumer diet sums if True
    DC_tol=0.001,             # Allowed deviation of diet sums from 1
)

C:\Users\idoca\Desktop\אישי\אקדמיה\תואר שני\מחקר\BTN\GlobalPPREstimation\tools\scientific_code\PPREstimation\PPRCalculator.py:328: RuntimeWarning: Diet normalization compensated through runtime biomass accumulation for 43 Regular/PP group(s); computational closure, not observed biomass change. EE, M0 and detritus flows retained; see diet_normalization_balance_ledger.
  warnings.warn(


In [16]:
balanced, p, q = calculator.is_model_balanced()
balanced

In [5]:
# 100 * ((p - calculator.p) / calculator.p).rename(index=calculator.seq2name)

# DC

In [6]:
model.DC.sum(axis=1).rename(index=model.seq2name).round(decimals=7)
# model.DC.rename(index=model.seq2name, columns=model.seq2name).round(decimals=7).columns
# model.DC.rename(index=model.seq2name, columns=model.seq2name).round(decimals=7).loc['Pelagic sharks and rays']

diet_import              0.0
Detritus                 0.0
Producteurs primaires    0.0
Zool S                   1.0
Zool L                   1.0
Benthos                  1.0
Crustacés                1.0
Céphalopodes             1.0
Mésopélagiques           1.0
Bathy SM inv             1.0
Bathy S pred             1.0
Bathy L pred             1.0
Dem S inv                1.0
Dem S pred               1.0
Dem M inv                1.0
Dem M pred               1.0
Dem L inv                1.0
Dem L pred               1.0
Côtier S inv             1.0
Capitaine royal          1.0
Côtier S pred            1.0
Côtier M inv             1.0
Petit capitaine          1.0
Mulets                   1.0
Disques                  1.0
Mâchoiron                1.0
Côtier M pred            1.0
Côtier L pred            1.0
Gros capitaine           1.0
Bars                     1.0
Pel S inv                1.0
Pel L pred               1.0
Ethmalose                1.0
Côtier pel S inv         1.0
Côtier pel L p

In [9]:
# calculator.get_DC().sum(axis=1).round(decimals=3)

# Groups Data

In [10]:
# model.groups_data.rename(index=model.seq2name, columns=model.seq2name).round(decimals=7)

In [12]:
# calculator.get_groups_df().rename(index=model.seq2name, columns=model.seq2name).round(decimals=7)

# Diagnostics

In [11]:
result, sppr, A, L = calculator.diagnose_sppr(
    TE_option='TE',
    return_sppr=True
)

In [12]:
result['model_input']

{'status': 'OK',
 'is_model_balanced': True,
 'p_max_rel_residual': 9.051424625991331e-08,
 'q_max_rel_residual': 9.453343315609422e-08,
 'dc_rows_sum_to_1': True,
 'dc_max_deviation': 4.440892098500626e-16,
 'n_negative_catch': 0,
 'n_zero_catch': 17,
 'total_catch': 0.8679521626500001,
 'has_catch': True,
 'has_ee_issues': False,
 'n_ee0': 0,
 'n_ee_marginal': 0,
 'n_ee_gt_1': 0,
 'ee0_groups': [],
 'ee_marginal_groups': []}

In [13]:
result['divergence']

{'status': 'WARN',
 'solve_error': None,
 'b': 0.0,
 'b_converges': True,
 'rho_living': 0.7175712783782491,
 'living_converges': True,
 'sppr_det': {44: 0.42594771842232143},
 'max_sppr_det': 0.42594771842232143,
 'max_sppr_group': {'seq': 5,
  'tl': 4.474850729277922,
  'sppr': 630808.944285401,
  'inv_te': 33.31034482758621},
 'max_tl_group': {'seq': 2,
  'tl': 4.6569808764085865,
  'sppr': 517730.4369800872,
  'inv_te': 445.91662875671017},
 'n_negative_sources': 0,
 'expect_negatives': False,
 'near_singular_te': []}

In [14]:
result['balance']

{'status': 'OK',
 'is_balanced': True,
 'inflow': 4116.021785841957,
 'outflow': 4116.021626351154,
 'rel_gap': 3.874877535256206e-08}

In [15]:
result['footprint']

{'ppr_all': 3404.5240306719934,
 'ppr_inner': 3404.502244828889,
 'ppr_pp_only': 2758.6201358795242,
 'npp': 4116.0,
 'ppr2npp': 0.827138543447252,
 'ppr2npp_pp_only': 0.6702186919046463}

In [16]:
result['config']

{'TE_option': 'TE',
 'det_open_mode': 'none',
 'det_theta': 1.0,
 'det_external_sppr': 0.0,
 'det_collapse_mode': 'never',
 'explicit_TE': False,
 'method': None,
 'would_pool': False,
 'model': 'Guinea Current (1998)'}

In [17]:
result['warnings']

["TE_option='TE' has no detritus recycling matrix: b reported as 0.0 (mortality-derived SPPR is written off as lost)",
 'rho(A_LL)=0.7176 is within 0.3 of divergence (max SPPR 6.31e+05 at group 5 (Sélaciens L prof), TL 4.47)']